<a href="https://colab.research.google.com/github/rkk7452/OVRO-source-matching/blob/main/Animated_PyBDSF_Match_Sources_73MHz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Key Info:

Matching Sources notebook

This notebook relies on:
/content/drive/MyDrive/Caltech SRC '26/combined_sources_73MHz.csv

This file outputs:
/content/drive/MyDrive/Caltech SRC '26/Matching_Source_Pairs_73MHz.mp4
  video of the matching source pair by pair (commented out now, last exported for version 2 of matching)


  Version 3 source matching outputs:

  /content/drive/MyDrive/Caltech SRC '26/73_MHz_Source_Tracking_2.csv
    csv of full master catalog (for version 3 of matching)
  /content/drive/MyDrive/Caltech SRC '26/73_MHz_Source_Tracking_Detection_log_2.csv
    csv of each source that was matched (detection log df in version 3)

  Old version previously output:

  /content/drive/MyDrive/Caltech SRC '26/73_MHz_Source_Tracking.csv
    csv of full master catalog (for version 3 of matching)
  /content/drive/MyDrive/Caltech SRC '26/73_MHz_Source_Tracking_Detection_log.csv
    csv of each source that was matched (detection log df in version 3)

"""

In [ ]:
# Import necessary packages
import numpy as np
import matplotlib.pyplot as plt
import astropy
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization import ZScaleInterval, ImageNormalize
from astropy.coordinates import SkyCoord
import astropy.units as u
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

from matplotlib.ticker import MultipleLocator # custom ticks in plots
import colorcet as cc # more colormaps
from scipy.interpolate import griddata # interpolation for TEC background (old)
from scipy.interpolate import RBFInterpolator # interpolation for TEC background
from datetime import datetime
import pytz
from scipy.spatial import cKDTree # for limiting interpolation to concentrated regions

In [ ]:
# Set up messaging notification function (AI but just for fun)
import requests
WEBHOOK = "https://discord.com/api/webhooks/1527533400951488642/86K9hCZZ5hUc-P28cBQEZ12Jzp0ymT-k_7PB-3D4eKmkmfeXLCiPFaliF4scdIG6ZrDl"

def notify(message):
  r = requests.post(
      WEBHOOK,
      json={"content": message}
  )

  if r.status_code == 204:
      print("Notification sent!")
  else:
      print("Error:", r.status_code, r.text)


In [ ]:
# Needed if running on Google Colab only
# Gives access to data stored on Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


In [ ]:
date = "20250508"
freq = 64

folder_path = "/content/drive/MyDrive/Caltech SRC '26/OVRO-LWA Data/" # where raw fits.fz files are
ryan_folder_path = f"/content/drive/MyDrive/Caltech SRC '26/Output/{date}/" # where to export plots and data to
deep_int_folder = "/content/drive/MyDrive/Caltech SRC '26/Deep-Integrated-Sources-Catalog/" # where the deep int source files are
short_int_folder = "/content/drive/MyDrive/Caltech SRC '26/Short-Integrated-Sources-Catalog/" # where the short int source files are


In [ ]:
df = pd.read_csv(os.path.join(ryan_folder_path, short_int_folder, date, f"combined_sources_{freq}MHz_{date}.csv"))
df = df.sort_values(by='file_name',ascending=True)
#print(df.head())

In [ ]:
#set up

unique_files = df['file_name'].unique()

###Set up Matching Hard Thresholds###
seplimit = 3 * u.arcmin # sources can be a max of 3 arc min apart to be considered a match
fluxlimit = 0.2 # 20% limit in flux difference between sources (not being used as of 7/14/2026)

In [ ]:
###temp###

#Look at elevation data
df = df.sort_values(by='elevation')
display(df.head())

fig, ax = plt.subplots()
x = df['elevation']


bins = np.linspace(-10, 100, 100) # x min, x max, bins

plt.hist(x, bins, alpha=0.5, label='Source Elevation', color='blue')


plt.xlabel("Elevation (deg)")
plt.ylabel("# of Occurances")
plt.xlim(-10,100)
ax.xaxis.set_major_locator(MultipleLocator(5))
ax.xaxis.set_minor_locator(MultipleLocator(1))
plt.legend(loc='upper right')
plt.title("Source Elevation")
plt.show()

In [ ]:
# set up master catalog and detections df's


master_catalog = pd.read_csv(os.path.join(deep_int_folder,date,f"{freq}MHz_{date}_Deep_Integration_Sources.csv"))

master_catalog.insert(0, 'master_id', range(0, len(master_catalog)))

detections = pd.DataFrame(columns=['master_id', 'source_num', 'ra', 'dec', 'total_flux', 'elevation', 'file_name'])

master_catalog = master_catalog.set_index("master_id")
master_catalog['num_detections'] = 0
master_catalog = master_catalog.rename(columns={'ra': 'long_ra', 'dec': 'long_dec', 'total_flux': 'long_flux'})

display(master_catalog.head())
print(f'Length of Master Catalog: {len(master_catalog)}')

master_coords = SkyCoord(ra=master_catalog['long_ra'].values*u.deg, dec=master_catalog['long_dec'].values*u.deg) # first set to match from


In [ ]:
#Filter main df and master catalog to sources sufficiently above the horizon:

print(len(df))

min_elevation = 5 # anything lower is removed from the dataset---deletes below horizon anomalies and horizon-level contamination

df = df[df['elevation'] > min_elevation].copy()# drop values <= min_elevation
master_catalog = master_catalog[master_catalog['elevation'] > min_elevation].copy()

print(len(df))

In [ ]:
"""
Version 3
Algorithm to track and master_id sources across multiple files
"""



for item in unique_files:
  file_2 = df[df['file_name']==item]
  coords_list = SkyCoord(ra=file_2['ra'].values*u.deg, dec=file_2['dec'].values*u.deg) # creats the SkyCoord object from the df
  idx_1, idx_2, d2d, d3d = astropy.coordinates.search_around_sky(master_coords, coords_list, seplimit)
  matches_df = pd.DataFrame({
    'f1_source_id': master_catalog.index[idx_1],

    'f2_source_id': file_2.iloc[idx_2]['source_id'].values,
    'f2_ra': file_2.iloc[idx_2]['ra'].values,
    'f2_dec': file_2.iloc[idx_2]['dec'].values,
    'f2_flux': file_2.iloc[idx_2]['total_flux'].values,
    'f2_elevation': file_2.iloc[idx_2]['elevation'].values,

    'separation_arcmin': d2d.to(u.arcmin).value
  })


  #rank and choose only the best new source to match a given source in the master catalog:

  matches_df["flux_ratio"] = matches_df["f2_flux"] / master_catalog.loc[matches_df["f1_source_id"], "long_flux"].values #create flux ratio

  #drop matches with higher than fluxlimit flux ratio
  #matches_df = matches_df[(matches_df["flux_ratio"] - 1).abs() <= fluxlimit]

  # Calculate how much brighter this short integration file is compared to the master catalog on average
  # We use the median of all initial candidates to avoid being skewed by outliers or bad matches
  median_scale = np.median(matches_df["f2_flux"] / master_catalog.loc[matches_df["f1_source_id"], "long_flux"].values)

  # Normalize the flux ratio by this scale factor
  matches_df["normalized_flux_ratio"] = matches_df["flux_ratio"] / median_scale

  # Now, a perfect match will have a normalized_flux_ratio of 1.0
  matches_df["score"] = (matches_df["separation_arcmin"] / seplimit.to(u.arcmin).value)
  #+ 0.5 * np.abs(np.log(matches_df["normalized_flux_ratio"])) # take out for now b/c flux is different scales
  #scores a given match:
    #the separation in arc minutes is penalized, where 0 is a perfect match (same location) and 1 is the exact value of seplimit (furthest away possible to be considered a match)
    #the flux ratio is also penzalized. If they have the same flux, the ratio is 1 and log(1) = 0, so there's no penalty
    #then they're added together to create score, in which the lower the score, the better the match

  matches_df = (matches_df.sort_values("score").drop_duplicates(subset="f1_source_id", keep="first").drop_duplicates(subset='f2_source_id',keep='first'))
    # only keeps the best scored match on each end (1-1 mapping)

  #update the master catalog:
  for _, match in matches_df.iterrows():
    master_id = match["f1_source_id"]

    n = master_catalog.loc[master_id, "num_detections"]
    master_catalog.loc[master_id, "num_detections"] = n + 1 # update num detections

  #update the detections df:

  new_rows = []# create empty list

  for _, match in matches_df.iterrows(): # loop through each row in the matches_df
    new_rows.append( {
        "master_id": match['f1_source_id'],
        'source_num': match['f2_source_id'],
        'ra': match['f2_ra'],
        'dec': match['f2_dec'],
        'total_flux': match['f2_flux'],
        'elevation': match['f2_elevation'],
        'file_name': item
    })

  #add the new_rows list to the detections df
  detections = pd.concat([detections, pd.DataFrame(new_rows)], ignore_index=True)

  #update master_coords
  master_coords = SkyCoord(
    ra=master_catalog["long_ra"].values * u.deg,
    dec=master_catalog["long_dec"].values * u.deg
  )


In [ ]:
###temp###

#Flux distribution

### with matches, not filtered
x = master_catalog['long_flux']
y = df['total_flux']
z = detections['total_flux']

bins = np.linspace(0, 100, 100) # x min, x max, bins

plt.hist(x, bins, alpha=0.5, label='Deep Integration', color='blue')
plt.hist(y, bins, alpha=0.5, label='Short Integration', color='green')
plt.hist(z, bins, alpha=0.3, label='Matches',color='red')


plt.xlabel("Flux")
plt.ylabel("# of Occurances")

plt.legend(loc='upper right')
plt.title(f"{date} {freq} MHz Flux Distribution With Matches (before filtering)")
plt.savefig(os.path.join(ryan_folder_path, f"{date}_{freq}MHz_Flux_Distribution_With_Matches_No_Filter.png"))
plt.show()


### without matches, not filtered


x = master_catalog['long_flux']
y = df['total_flux']
z = detections['total_flux']

bins = np.linspace(0, 100, 100) # x min, x max, bins

plt.hist(x, bins, alpha=0.5, label='Deep Integration', color='blue')
plt.hist(y, bins, alpha=0.5, label='Short Integration', color='green')


plt.xlabel("Flux")
plt.ylabel("# of Occurances")

plt.legend(loc='upper right')
plt.title(f"{date} {freq} MHz Flux Distribution Without Matches (before filtering)")
plt.savefig(os.path.join(ryan_folder_path, f"{date}_{freq}MHz_Flux_Distribution_Without_Matches_No_Filter.png"))
plt.show()



In [ ]:
#Filter Results

min_detections = 2 #### can change this number ####
min_flux = 0 #### can change this number, median was 10.4. Ruby says 1-5 seems good, need to adjust. ####

print(len(master_catalog))

master_catalog = master_catalog[master_catalog['num_detections'] > min_detections].copy()# drop values <= min_detections
master_catalog = master_catalog[master_catalog['long_flux'] > min_flux].copy()# drop values <= min_flux

detections = detections[detections['master_id'].isin(master_catalog.index)].copy() # make sure detections is up to date

#display(master_catalog.head())
#display(master_catalog.tail())

print(len(master_catalog))

# clean up ra values to make it 0-360 instead of -360 to 0
master_catalog['long_ra'] = master_catalog['long_ra'] % 360
detections['ra'] = detections['ra'] % 360

In [ ]:
#now...export results!

master_catalog = master_catalog.sort_values(by='num_detections', ascending=False)
display(master_catalog.head())
master_catalog.to_csv(os.path.join(ryan_folder_path,f"{date}_{freq}MHz_Master_Catalog.csv"), index = True, index_label = 'master_id')
detections.to_csv(os.path.join(ryan_folder_path, f"{date}_{freq}MHz_Detection_Log.csv"), index = False)

In [ ]:
###temp###

### with matches, after filtering
x = master_catalog['long_flux']
y = df['total_flux']
z = detections['total_flux']

bins = np.linspace(0, 100, 100) # x min, x max, bins

plt.hist(x, bins, alpha=0.5, label='Deep Integration', color='blue')
plt.hist(y, bins, alpha=0.5, label='Short Integration', color='green')
plt.hist(z, bins, alpha=0.3, label='Matches',color='red')


plt.xlabel("Flux")
plt.ylabel("# of Occurances")

plt.legend(loc='upper right')
plt.title(f"{date} {freq}MHz Flux Distribution With Matches (after filtering)")
plt.savefig(os.path.join(ryan_folder_path, f"{date}_{freq}MHz_Flux_Distribution_With_Matches_Filtered.png"))
plt.show()

In [ ]:
"""

Next Steps:

find other ovro lwa papers and see how low of an elevation they used as the cutoff (mountains + aritificial interference at low elevation sources)

[done] convert differential STEC to differential TEC
convert differential TEC to TEC


[done?] map the vector offsets into actual ionospheric tec units
        wouldn't just be a magnitude but also direction
        equations in the papers we read
          can use the Jordan et al paper equation for the STEC equation

Then later steps:

[done] have a saved animation for all frequencies across all times
[done] interpolation from individual sources and overall structure
[done] going from source offsets to estimate of differential TEC

Ruby's not sure if should do TEC or interpolation first

[waiting] look at another night of data
    looks like not a lot of coherent ionospheric structure on this night so far

Interpolation explanation:
  predicting the more continuous fill based on trend of nearby data

Maybe try to find out the total TEC in our data
  don't measure this b/c the offsets correspond to only differential TEC
  can look up satellite data to get total TEC estimates
    something other ppl in the dept could look at as well
    US Naval Research takes these measurements and using that, can model the total ionospheric TEC

  offsets give us the derivative of the TEC b/c they are changing. We can use that to infer what the full TEC is:
    integral of differential TEC
    we can get the slopes but we would need a satellite measurement of total TEC to get the actual y offset / scale of the integral of the differential TEC

"""

In [ ]:
### Newest Version: Animation + Offset calculations, but separated ###

# This cell is the offset calculation section

"""FYI
combined_offsets is a list
each iteration of the loop, the temporary offset_df is added to the list
after the loop, combined_offsets is converted to the df total_offsets_df

Equation for conversion to STEC based off of Jordan et al. '18
https://arxiv.org/pdf/1707.04978

Equation for STEC to TEC conversion based off of Ja Soon Shim '09 citing Mannucci et al., 1993
https://digitalcommons.usu.edu/cgi/viewcontent.cgi?article=1418&context=etd

"""

#Constants set up for TECU conversion:

e_c = 1.602176634e-19       # Electron charge [C]
m_e = 9.1093837015e-31    # Electron mass [kg]
eps_0 = 8.8541878128e-12  # Vacuum permittivity [F/m]
nu = 73.0e6               # Observation frequency [Hz]

iono_shell_height = 450 # the height of the thin-shell ionosphere model (km)
earth_mean_radius = 6371 # the mean radius of the earth (km)

# Complete unsimplified constant factor (m^2 / rad)
constant_factor = -(1.0 / (8.0 * np.pi**2)) * (e_c**2 / (eps_0 * m_e)) * (1.0 / nu**2)

# Conversion factor from raw electrons/m^2 to TECU
TECU_CONVERSION = 1e16


combined_offsets = [] # empty list, will add to it each iteration of the loop


for i in range(0, len(unique_files)):

  file_to_check = unique_files[i] # the name of the nth unique file name in the data

  # create smaller df's with only the sources in the file we're looking at
  detections_sub = detections[detections['file_name']==file_to_check].copy() # detections log subset

  #new smaller df with detections_sub and master catalog based on master id
  merged_sub = detections_sub.merge(
      master_catalog,
      left_on='master_id',
      right_index=True,
      suffixes=('', '_master')
  )

  if merged_sub.empty:
    continue


  ra_raw_offset = merged_sub['ra'] - merged_sub['long_ra']
  dec_offset_deg = merged_sub['dec'] - merged_sub['long_dec']

  ra_offset_deg = (ra_raw_offset + 180) % 360 - 180 # still in degrees, allowing offsets to wrap around the sides
  ra_offset_deg = ra_offset_deg * np.cos(np.radians(merged_sub['long_dec'])) # correct for curvature at high/low dec

  #convert to radians
  theta_RA_rad = np.radians(ra_offset_deg)
  theta_Dec_rad = np.radians(dec_offset_deg)

  #calculate raw diff STEC gradients (electrons/m^2/rad)
  STEC_RA_raw = theta_RA_rad / constant_factor
  STEC_Dec_raw = theta_Dec_rad / constant_factor

  #convert to diff TECU/rad
  STEC_RA_TECU = STEC_RA_raw / TECU_CONVERSION
  STEC_Dec_TECU = STEC_Dec_raw / TECU_CONVERSION

  STEC_mag_TECU = np.sqrt(STEC_RA_TECU**2 + STEC_Dec_TECU**2) # find overall TEC using sqrt Pythagorean theorem

  offset_df = pd.DataFrame({
      'master_id': merged_sub['master_id'],
      'ra': merged_sub['ra'],
      'dec': merged_sub['dec'],
      'ra_offset': ra_offset_deg, # needed to draw vectors?
      'dec_offset': dec_offset_deg, # needed to draw vectors?
      'flux_offset': merged_sub['total_flux'] - merged_sub['long_flux'], # in degrees
      'offset_dir': np.degrees(np.atan2(dec_offset_deg, ra_offset_deg)),

      # New Ionospheric Gradients in TECU/rad
      'STEC_RA': STEC_RA_TECU,
      'STEC_Dec': STEC_Dec_TECU,
      'STEC_mag': STEC_mag_TECU,
      'file_name': file_to_check
  })

  # get the VTEC:
  elevation_series = offset_df['master_id'].map(detections['elevation'])# Map the elevation from master_catalog using the master_id index
  offset_df['TEC'] = offset_df['STEC_mag'] * (1 - ((np.cos(np.radians(elevation_series)))/(1+iono_shell_height/earth_mean_radius))**2)**0.5 # Calculate the TEC using the mapped elevation
  offset_df['TEC_RA'] = offset_df['STEC_RA'] * (1 - ((np.cos(np.radians(elevation_series)))/(1+iono_shell_height/earth_mean_radius))**2)**0.5 # Calculate the TEC using the mapped elevation
  offset_df['TEC_Dec'] = offset_df['STEC_Dec'] * (1 - ((np.cos(np.radians(elevation_series)))/(1+iono_shell_height/earth_mean_radius))**2)**0.5 # Calculate the TEC using the mapped elevation

  combined_offsets.append(offset_df) # add to combined_offsets


total_offsets_df = pd.concat(combined_offsets, ignore_index=True) # makes it into a df


In [ ]:
###temp###

print(offset_df['TEC'].max())

In [ ]:
###temp###

display(offset_df.head())

In [ ]:
# Part 2: The animation part, using the offset df created above

#without background

#set up

fig, ax = plt.subplots(figsize = (16,9), facecolor = 'black') # black outside background
base_q = ax.quiver([0], [0], [0], [0], [0], cmap=cc.cm['CET_C7'])
base_q.set_clim(-180, 180)

cbar = fig.colorbar(base_q)
cbar.set_label('Offset Direction (Degrees)', color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')

ax.clear()

#animation loop
def animate (frame_index):
  ax.clear()

  #re apply stylings that are cleared by ax.clear()
  ax.set_facecolor('black') # black background inside the plot
  ax.set_xlabel("RA", color='white', fontsize=12) # label axis
  ax.set_ylabel("Dec", color='white', fontsize=12) # label axis
  ax.tick_params(colors='white') # white ticks
  ax.set_xlim(0, 360) # range of x axis in degrees (RA)
  ax.set_ylim(-90, 90) # range of y axis in degrees (DEC)

  file_to_check = unique_files[frame_index]
  ax.set_title(f"Vector Offsets for {file_to_check}", color='white', fontsize=14) # title plot

  frame_df = total_offsets_df[total_offsets_df['file_name'] == file_to_check]
  if frame_df.empty:
        print(f"Skipping Frame: {frame_index}. File_name: {file_to_check}")
        return
  frame_df = frame_df.dropna(subset=['ra', 'dec', 'TEC']).copy() # remove any NaN values if they exist for any reason

  #draw vectors:

  #magnitudes = np.hypot(frame_df['ra_offset'], frame_df['dec_offset']) # Option 1: color is based off of arrow magnitude
  magnitudes = frame_df['offset_dir'] # Option 2: color is based off of arrow direction

  q = ax.quiver(
      frame_df['ra'],
      frame_df['dec'],
      frame_df['ra_offset'],
      frame_df['dec_offset'],
      magnitudes,
      angles='xy',
      scale_units='xy',
      scale=0.006 ,      # smaller num is larger arrow length
      width=0.002,      # larger num is wider arrow shaft
      headwidth=5,
      headlength=5,
      cmap=cc.cm['CET_C7'])

ani = FuncAnimation(fig, animate, frames=len(unique_files))
ani.save(os.path.join(ryan_folder_path, f"{date}_{freq}MHz_Vector_Animation_No_Background.mp4"), writer='ffmpeg', fps=2, dpi=224)

notify(f'{freq}MHz No-Background animation done! {datetime.now(pytz.timezone("US/Pacific")).strftime("%b %d, %Y at %H:%M:%S")}')


In [ ]:
# Part 2a: The animation part, using the offset df created above

#with background, background is overall TEC magnitude

#set up plot

fig, ax = plt.subplots(figsize=(16, 9), facecolor='black', layout='constrained')

min = 0
max = 0.0001

# Colorbar 1 (for the interpolation (background colors))
mappable_mesh = plt.cm.ScalarMappable(
    norm=plt.Normalize(vmin=min, vmax=max),
    cmap='plasma'
)

cbar_mesh = fig.colorbar(mappable_mesh, ax=ax, pad=0.00, shrink=0.6)
cbar_mesh.set_label('Background - TEC Magnitude', color='white', fontsize=12)
cbar_mesh.ax.yaxis.set_tick_params(color='white', labelcolor='white')

# Static Colorbar 2: For the Vector Offsets (arrow colors)
mappable_quiver = plt.cm.ScalarMappable(
    norm=plt.Normalize(vmin=-180, vmax=180),
    cmap=cc.cm['CET_C7']
)
cbar_quiv = fig.colorbar(mappable_quiver, ax=ax, pad=0.02, shrink=0.6)
cbar_quiv.set_label('Vectors - Offset Direction (Deg)', color='white', fontsize=12)
cbar_quiv.ax.yaxis.set_tick_params(color='white', labelcolor='white')



# set up interpolation grid

ra = np.linspace(0, 360, 250)
dec = np.linspace(-90, 90, 250)
ra_grid, dec_grid = np.meshgrid(ra, dec, indexing='xy')

# Flatten the grid into a (62500, 2) array for the interpolator
grid_points = np.column_stack((ra_grid.ravel(), dec_grid.ravel()))

#animation loop
def animate (frame_index):
  ax.clear()

  #re apply stylings that are cleared by ax.clear()
  ax.set_facecolor('black') # black background inside the plot
  ax.set_xlabel("RA", color='white', fontsize=12) # label axis
  ax.set_ylabel("Dec", color='white', fontsize=12) # label axis
  ax.tick_params(colors='white') # white ticks
  ax.set_xlim(0, 360) # range of x axis in degrees (RA)
  ax.set_ylim(-90, 90) # range of y axis in degrees (DEC)

  file_to_check = unique_files[frame_index]
  ax.set_title(f"Vector Offsets for {file_to_check}", color='white', fontsize=14) # title plot

  frame_df = total_offsets_df[total_offsets_df['file_name'] == file_to_check]
  if frame_df.empty:
        print(f"Skipping Frame: {frame_index}. File_name: {file_to_check}")
        return

  frame_df = frame_df.dropna(subset=['ra', 'dec', 'TEC']).copy() # remove any NaN values if they exist for any reason

  #create background based off of offset magnitude

  obs_coords = frame_df[['ra', 'dec']].to_numpy()  # Shape: (N, 2)
  obs_values = frame_df['TEC'].to_numpy()          # Shape: (N,)
  rbf = RBFInterpolator(obs_coords, obs_values)

  # Evaluate the interpolator on the grid
  grid_values_flat = rbf(grid_points)

  # Compute distance from each grid point to the nearest observation
  tree = cKDTree(obs_coords)
  dist, idx = tree.query(grid_points, k=2)

  mask = dist[:, 1] <= 12 # must be within 12 degrees of 2 sources

  # Mask out regions too far from any observation
  grid_values_flat[~mask] = np.nan

  # Convert back to a 2D grid
  ygrid = grid_values_flat.reshape(ra_grid.shape)

  mesh = ax.pcolormesh(
      ra_grid,
      dec_grid,
      ygrid,
      vmin=min,
      vmax=max,
      shading='gouraud',
      cmap='plasma',
      zorder=1
  )

  #draw vectors:

  #magnitudes = np.hypot(frame_df['ra_offset'], frame_df['dec_offset']) # Option 1: color is based off of arrow magnitude
  magnitudes = frame_df['offset_dir'] # Option 2: color is based off of arrow direction

  q = ax.quiver(
      frame_df['ra'],
      frame_df['dec'],
      frame_df['ra_offset'],
      frame_df['dec_offset'],
      magnitudes,
      angles='xy',
      scale_units='xy',
      scale=0.006 ,      # smaller num is larger arrow length
      width=0.002,      # larger num is wider arrow shaft
      headwidth=5,
      headlength=5,
      clim=(-180, 180),
      cmap=cc.cm['CET_C7'],
      alpha=0.8,
      zorder=2)

ani = FuncAnimation(fig, animate, frames=len(unique_files))
ani.save(os.path.join(ryan_folder_path, f"{date}_{freq}MHz_Vector_Animation_Background_Magnitude.mp4"), writer='ffmpeg', fps=2, dpi=224)

notify(f'{freq}MHz {date} TEC Background animation done! {datetime.now(pytz.timezone("US/Pacific")).strftime("%b %d, %Y at %H:%M:%S")}')


In [ ]:
# Part 2b: The animation part, using the offset df created above

#with background, background is RA TEC magnitude

#set up plot

fig, ax = plt.subplots(figsize=(16, 9), facecolor='black', layout='constrained')

min = 0
max = 0.0001

# Colorbar 1 (for the interpolation (background colors))
mappable_mesh = plt.cm.ScalarMappable(
    norm=plt.Normalize(vmin=min, vmax=max),
    cmap='plasma'
)

cbar_mesh = fig.colorbar(mappable_mesh, ax=ax, pad=0.00, shrink=0.6)
cbar_mesh.set_label('Background - TEC RA Factor Magnitude', color='white', fontsize=12)
cbar_mesh.ax.yaxis.set_tick_params(color='white', labelcolor='white')

# Static Colorbar 2: For the Vector Offsets (arrow colors)
mappable_quiver = plt.cm.ScalarMappable(
    norm=plt.Normalize(vmin=-180, vmax=180),
    cmap=cc.cm['CET_C7']
)
cbar_quiv = fig.colorbar(mappable_quiver, ax=ax, pad=0.02, shrink=0.6)
cbar_quiv.set_label('Vectors - Offset Direction (Deg)', color='white', fontsize=12)
cbar_quiv.ax.yaxis.set_tick_params(color='white', labelcolor='white')



# set up interpolation grid

ra = np.linspace(0, 360, 250)
dec = np.linspace(-90, 90, 250)
ra_grid, dec_grid = np.meshgrid(ra, dec, indexing='xy')

# Flatten the grid into a (62500, 2) array for the interpolator
grid_points = np.column_stack((ra_grid.ravel(), dec_grid.ravel()))

#animation loop
def animate (frame_index):
  ax.clear()

  #re apply stylings that are cleared by ax.clear()
  ax.set_facecolor('black') # black background inside the plot
  ax.set_xlabel("RA", color='white', fontsize=12) # label axis
  ax.set_ylabel("Dec", color='white', fontsize=12) # label axis
  ax.tick_params(colors='white') # white ticks
  ax.set_xlim(0, 360) # range of x axis in degrees (RA)
  ax.set_ylim(-90, 90) # range of y axis in degrees (DEC)

  file_to_check = unique_files[frame_index]
  ax.set_title(f"Vector Offsets for {file_to_check}", color='white', fontsize=14) # title plot

  frame_df = total_offsets_df[total_offsets_df['file_name'] == file_to_check]
  if frame_df.empty:
        print(f"Skipping Frame: {frame_index}. File_name: {file_to_check}")
        return

  frame_df = frame_df.dropna(subset=['ra', 'dec', 'TEC']).copy() # remove any NaN values if they exist for any reason

  #create background based off of offset magnitude

  obs_coords = frame_df[['ra', 'dec']].to_numpy()  # Shape: (N, 2)
  obs_values = frame_df['TEC_RA'].to_numpy()          # Shape: (N,)
  rbf = RBFInterpolator(obs_coords, obs_values)

  # Evaluate the interpolator on the grid
  grid_values_flat = rbf(grid_points)

  # Compute distance from each grid point to the nearest observation
  tree = cKDTree(obs_coords)
  dist, idx = tree.query(grid_points, k=2)

  mask = dist[:, 1] <= 12 # must be within 12 degrees of 2 sources

  # Mask out regions too far from any observation
  grid_values_flat[~mask] = np.nan

  # Convert back to a 2D grid
  ygrid = grid_values_flat.reshape(ra_grid.shape)

  mesh = ax.pcolormesh(
      ra_grid,
      dec_grid,
      ygrid,
      vmin=min,
      vmax=max,
      shading='gouraud',
      cmap='plasma',
      zorder=1
  )

  #draw vectors:

  #magnitudes = np.hypot(frame_df['ra_offset'], frame_df['dec_offset']) # Option 1: color is based off of arrow magnitude
  magnitudes = frame_df['offset_dir'] # Option 2: color is based off of arrow direction

  q = ax.quiver(
      frame_df['ra'],
      frame_df['dec'],
      frame_df['ra_offset'],
      frame_df['dec_offset'],
      magnitudes,
      angles='xy',
      scale_units='xy',
      scale=0.006 ,      # smaller num is larger arrow length
      width=0.002,      # larger num is wider arrow shaft
      headwidth=5,
      headlength=5,
      clim=(-180, 180),
      cmap=cc.cm['CET_C7'],
      alpha=0.8,
      zorder=2)

ani = FuncAnimation(fig, animate, frames=len(unique_files))
ani.save(os.path.join(ryan_folder_path, f"{date}_{freq}MHz_Vector_Animation_Background_RA.mp4"), writer='ffmpeg', fps=2, dpi=224)

notify(f'{freq}MHz {date} RA TEC Background animation done! {datetime.now(pytz.timezone("US/Pacific")).strftime("%b %d, %Y at %H:%M:%S")}')


In [ ]:
# Part 2c: The animation part, using the offset df created above

#with background, background is Dec TEC magnitude

#set up plot

fig, ax = plt.subplots(figsize=(16, 9), facecolor='black', layout='constrained')

min = 0
max = 0.0001

# Colorbar 1 (for the interpolation (background colors))
mappable_mesh = plt.cm.ScalarMappable(
    norm=plt.Normalize(vmin=min, vmax=max),
    cmap='plasma'
)

cbar_mesh = fig.colorbar(mappable_mesh, ax=ax, pad=0.00, shrink=0.6)
cbar_mesh.set_label('Background - TEC Dec Factor Magnitude', color='white', fontsize=12)
cbar_mesh.ax.yaxis.set_tick_params(color='white', labelcolor='white')

# Static Colorbar 2: For the Vector Offsets (arrow colors)
mappable_quiver = plt.cm.ScalarMappable(
    norm=plt.Normalize(vmin=-180, vmax=180),
    cmap=cc.cm['CET_C7']
)
cbar_quiv = fig.colorbar(mappable_quiver, ax=ax, pad=0.02, shrink=0.6)
cbar_quiv.set_label('Vectors - Offset Direction (Deg)', color='white', fontsize=12)
cbar_quiv.ax.yaxis.set_tick_params(color='white', labelcolor='white')



# set up interpolation grid

ra = np.linspace(0, 360, 250)
dec = np.linspace(-90, 90, 250)
ra_grid, dec_grid = np.meshgrid(ra, dec, indexing='xy')

# Flatten the grid into a (62500, 2) array for the interpolator
grid_points = np.column_stack((ra_grid.ravel(), dec_grid.ravel()))

#animation loop
def animate (frame_index):
  ax.clear()

  #re apply stylings that are cleared by ax.clear()
  ax.set_facecolor('black') # black background inside the plot
  ax.set_xlabel("RA", color='white', fontsize=12) # label axis
  ax.set_ylabel("Dec", color='white', fontsize=12) # label axis
  ax.tick_params(colors='white') # white ticks
  ax.set_xlim(0, 360) # range of x axis in degrees (RA)
  ax.set_ylim(-90, 90) # range of y axis in degrees (DEC)

  file_to_check = unique_files[frame_index]
  ax.set_title(f"Vector Offsets for {file_to_check}", color='white', fontsize=14) # title plot

  frame_df = total_offsets_df[total_offsets_df['file_name'] == file_to_check]
  if frame_df.empty:
        print(f"Skipping Frame: {frame_index}. File_name: {file_to_check}")
        return

  frame_df = frame_df.dropna(subset=['ra', 'dec', 'TEC']).copy() # remove any NaN values if they exist for any reason

  #create background based off of offset magnitude

  obs_coords = frame_df[['ra', 'dec']].to_numpy()  # Shape: (N, 2)
  obs_values = frame_df['TEC_Dec'].to_numpy()          # Shape: (N,)
  rbf = RBFInterpolator(obs_coords, obs_values)

  # Evaluate the interpolator on the grid
  grid_values_flat = rbf(grid_points)

  # Compute distance from each grid point to the nearest observation
  tree = cKDTree(obs_coords)
  dist, idx = tree.query(grid_points, k=2)

  mask = dist[:, 1] <= 12 # must be within 12 degrees of 2 sources

  # Mask out regions too far from any observation
  grid_values_flat[~mask] = np.nan

  # Convert back to a 2D grid
  ygrid = grid_values_flat.reshape(ra_grid.shape)

  mesh = ax.pcolormesh(
      ra_grid,
      dec_grid,
      ygrid,
      vmin=min,
      vmax=max,
      shading='gouraud',
      cmap='plasma',
      zorder=1
  )

  #draw vectors:

  #magnitudes = np.hypot(frame_df['ra_offset'], frame_df['dec_offset']) # Option 1: color is based off of arrow magnitude
  magnitudes = frame_df['offset_dir'] # Option 2: color is based off of arrow direction

  q = ax.quiver(
      frame_df['ra'],
      frame_df['dec'],
      frame_df['ra_offset'],
      frame_df['dec_offset'],
      magnitudes,
      angles='xy',
      scale_units='xy',
      scale=0.006 ,      # smaller num is larger arrow length
      width=0.002,      # larger num is wider arrow shaft
      headwidth=5,
      headlength=5,
      clim=(-180, 180),
      cmap=cc.cm['CET_C7'],
      alpha=0.8,
      zorder=2)

ani = FuncAnimation(fig, animate, frames=len(unique_files))
ani.save(os.path.join(ryan_folder_path, f"{date}_{freq}MHz_Vector_Animation_Background_Dec.mp4"), writer='ffmpeg', fps=2, dpi=224)

notify(f'{freq}MHz {date} Dec TEC Background animation done! {datetime.now(pytz.timezone("US/Pacific")).strftime("%b %d, %Y at %H:%M:%S")}')

In [ ]:
###temp###

# look into those offset results:
total_offsets_df = total_offsets_df.sort_values(by='ra_offset',ascending=True).copy()
display(total_offsets_df.head())

print(f"Length: {len(total_offsets_df)}")
print('\nOverview')
display(total_offsets_df[['ra_offset', 'dec_offset']].describe())
print("\nskew")
display(total_offsets_df[['ra_offset', 'dec_offset']].skew())
print("\nkurtosis")
display(total_offsets_df[['ra_offset', 'dec_offset']].kurtosis())

In [ ]:
###temp###

#plotting distribution histograms


total_offsets_df['ra_offset'].plot.hist(bins=100, figsize=(10, 6), title=f"{date} {freq}MHz RA Offsets Distribution")
plt.xlabel("Offset (Degrees)")
plt.ylabel("# of Occurances")
plt.xlim(-.05,.05)
plt.ylim(0,6200)
plt.savefig(os.path.join(ryan_folder_path, f"{date}_{freq}MHz_RA_Offsets_Distribution.png"))
plt.show()



total_offsets_df['dec_offset'].plot.hist(bins=100, figsize=(10, 6), title=f"{date} {freq}MHz Dec Offsets Distribution")
plt.xlabel("Offset (Degrees)")
plt.ylabel("# of Occurances")
plt.xlim(-.05,.05)
plt.ylim(0,6200)
plt.savefig(os.path.join(ryan_folder_path, f"{date}_{freq}MHz_Dec_Offsets_Distribution.png"))
plt.show()


In [ ]:
###temp###

#Plot changes over time

display(total_offsets_df.head())
display(total_offsets_df.tail())
std_ra_by_file = total_offsets_df.groupby('file_name')['ra_offset'].std().rename('Standard_Dev_ra_offset')
std_dec_by_file = total_offsets_df.groupby('file_name')['dec_offset'].std()

mean_dir_by_file = total_offsets_df.groupby('file_name')['offset_dir'].mean()


fig, ax = plt.subplots()

ax.plot(total_offsets_df['file_name'].unique(), std_ra_by_file, color='red')
ax.plot(total_offsets_df['file_name'].unique(), std_dec_by_file, color='green')

ax.set(ylim=(0,0.025))

ax.xaxis.set_major_locator(MultipleLocator(20))
ax.set_xlabel('Time')
ax.tick_params(labelbottom=False)


"""
ax.plot(total_offsets_df['file_name'].unique(), mean_dir_by_file, color='blue')

ax.set(ylim=(-90, 90))
ax.xaxis.set_major_locator(MultipleLocator(20))
ax.set_ylabel('Mean Offset Direction per Image (deg)')
plt.title('73 MHz Mean Offset Direction over time on 2025.05.07')
"""

plt.show()

In [ ]:
###temp###

std_ra_by_file = std_ra_by_file.sort_values(ascending=False).copy()
print("sorted!")
display(std_ra_by_file.head())

In [ ]:
###temp###
"""
#Plot specific figures with vectors

#set up to look for master id 638 right now

file_to_check = "73MHz-Clean-Snapshot-20250507_103142-image.fits"

# create smaller df's with only the sources in the file we're looking at
detections_sub = detections[detections['file_name']==file_to_check].copy() # detections log subset

#new smaller df with detections_sub and master catalog based on master id
merged_sub_new = detections_sub.merge(
    master_catalog,
    left_on='master_id',
    right_index=True,
    suffixes=('', '_master')
)

#print(f'Merged Detections & Catalog Subset. Len: {len(merged_sub_new)}') # check
#display(merged_sub_new.head())

offset_df_new = pd.DataFrame({
    'master_id': merged_sub_new['master_id'],
    'ra_raw_offset': merged_sub_new['ra'] - merged_sub_new['long_ra'], #raw difference in degrees
    'dec_offset': merged_sub_new['dec'] - merged_sub_new['long_dec'],
    'flux_offset': merged_sub_new['total_flux'] - merged_sub_new['long_flux'],
})
offset_df_new['ra_raw_offset'] = (offset_df_new['ra_raw_offset'] + 180) % 360 - 180
offset_df_new['ra_offset'] = offset_df_new['ra_raw_offset'] * np.cos(np.radians(merged_sub_new['long_dec'])) # corrected for curvature
offset_df_new['offset_dir'] = np.degrees(np.atan2(offset_df_new['dec_offset'], offset_df_new['ra_offset'])) # angle of the offset in degrees



offset_df_new['file_name'] = file_to_check # know which file it came from

#print(f'offset_df_new. Len: {len(offset_df_new)}')
#display(offset_df_new.head())
#display(offset_df_new.tail())
if 638 in offset_df_new['master_id'].values:
  print("638 is in this image")
else:
  print("638 is not in this image")

#plot with vectors:

fig, ax = plt.subplots(figsize = (15,10), facecolor = 'black') # black outside background
ax.set_facecolor('black') # black background inside the plot
ax.set_xlabel("RA", color='white', fontsize=12) # label axis
ax.set_ylabel("Dec", color='white', fontsize=12) # label axis
ax.tick_params(colors='white') # white ticks
ax.set_title(f"Vector Offsets for {file_to_check}", color='white', fontsize=14) # title plot


ax.set_xlim(0,360) # range of x axis in degrees (RA)
ax.set_ylim(-90, 90) # range of y axis in degrees (DEC)


# magnitudes = np.hypot(offset_df_new['ra_offset'], offset_df_new['dec_offset']) # old, color based off of magnitude of arrows
magnitudes = offset_df_new['offset_dir']
q = ax.quiver(merged_sub_new['ra'],
              merged_sub_new['dec'],
              offset_df_new['ra_offset'],
              offset_df_new['dec_offset'],
              magnitudes,
              angles='xy',
              scale_units='xy',
              scale=0.006 ,      # smaller num is larger arrow length
              width=0.002,      # larger num is wider arrow shaft
              headwidth=5,
              headlength=5,
              alpha = 0.2,
              cmap=cc.cm.CET_C8)

highlight_mask = offset_df_new['master_id'].isin([638, "638", 638.0])

if highlight_mask.any():
    # Filter coordinates and offsets for just 638
    sub_638 = merged_sub_new[highlight_mask]
    offset_638 = offset_df_new[highlight_mask]


    print("RA:", sub_638['ra'].values)
    print("Dec:", sub_638['dec'].values)

    # Overlay a thicker, brightly colored arrow on top
    ax.quiver(sub_638['ra'],
              sub_638['dec'],
              offset_638['ra_offset'],
              offset_638['dec_offset'],
              color='magenta',    # High contrast color against black and CET_C8
              angles='xy',
              scale_units='xy',
              scale=0.006,        # Keep scale same so lengths match relatively
              width=0.002,        # Made it 3x wider so it's noticeably thicker
              headwidth=6,
              headlength=6,
              alpha = 1,
              label='ID 638 Target')


cbar = fig.colorbar(q) # make a legend color bar
cbar.set_label('Offset Direction (Degrees)', color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')



plt.show()

"""

In [ ]:
notify(f"Completed {freq}MHz Matching & Animations! {datetime.now(pytz.timezone("US/Pacific")).strftime("%b %d, %Y at %H:%M:%S")}")